# 🩺 Huấn luyện Mamba-UNet (MambaVision Backbone) trên Google Colab

Quy trình toàn diện: Clone mã nguồn từ GitHub -> Cài đặt thư viện -> Mount Drive -> Giải nén chọn số lượng ca bệnh nhân vào SSD -> Huấn luyện -> Đánh giá 3D -> Lưu Checkpoint về Drive.

In [ ]:
# 1. Kiểm tra phần cứng và GPU khả dụng
!nvidia-smi

In [ ]:
# 2. Tải và đồng bộ mã nguồn từ GitHub
import os
import sys

GITHUB_REPO = 'https://github.com/NguyenMinhTri24072005/CT208E_LIDC_DuneAI_MambaVision.git'
BRANCH = 'main'  # Nhánh mặc định trên GitHub (đổi nếu bạn dùng nhánh khác)
REPO_DIR = '/content/CT208E_LIDC_DuneAI_MambaVision'

if os.path.exists('mamba_unet') and os.path.exists('common'):
    print(f'✅ Đang chạy trực tiếp trong thư mục dự án: {os.getcwd()}')
else:
    if not os.path.exists(REPO_DIR):
        print(f'📥 Đang clone repository từ GitHub (nhánh {BRANCH})...')
        !git clone -b {BRANCH} {GITHUB_REPO} {REPO_DIR}
    else:
        print(f'🔄 Đang đồng bộ mã nguồn mới nhất từ GitHub...')
        %cd {REPO_DIR}
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    %cd {REPO_DIR}

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'🚀 Thư mục làm việc hiện tại: {os.getcwd()}')

In [ ]:
# 3. Cài đặt các thư viện phụ thuộc
!pip install -q SimpleITK pandas scipy opencv-python-headless pyyaml matplotlib

In [ ]:
# 3b. (Tùy chọn) Cài đặt nhân CUDA mamba-ssm
# Lưu ý: Mamba-UNet hỗ trợ tự động Fallback thuần PyTorch nếu không cài được mamba-ssm
import torch
if torch.cuda.is_available():
    try:
        print('⚡ Thử cài đặt causal-conv1d và mamba-ssm để tăng tốc GPU...')
        !pip install -q causal-conv1d mamba-ssm
        print('✅ Cài đặt mamba-ssm thành công!')
    except Exception as e:
        print('⚠️ Sẽ sử dụng PyTorch selective_scan fallback mặc định:', e)

In [ ]:
# 4. Kết nối với Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 5. Chuẩn bị dữ liệu từ Google Drive vào SSD cục bộ (/content/data/cache_1mm)
# Bạn có thể chỉnh sửa NUM_PATIENTS: 0 = giải nén toàn bộ, hoặc nhập số ca (vd: 50, 100)
from common.data_pipeline.runtime_data import run_colab

# Đặt đường dẫn file zip trong Google Drive của bạn (sửa lại theo đúng đường dẫn của bạn)
DRIVE_ZIP_PATH = '/content/drive/MyDrive/cache_1mm.zip'  # Hoặc '/content/drive/MyDrive/DuneAI/cache_1mm.zip'
NUM_PATIENTS = 0  # 0: Toàn bộ dataset, >0: Chọn N bệnh nhân đầu tiên để test nhanh

cache_dir = run_colab(
    zip_path=DRIVE_ZIP_PATH,
    dest_dir='/content/data/cache_1mm',
    n_patients=NUM_PATIENTS,
)
print(f'✅ Dữ liệu sẵn sàng tại: {cache_dir}')

In [ ]:
# 6. Bắt đầu quá trình huấn luyện
!python -m mamba_unet.train_main train \
    --cache-dir /content/data/cache_1mm \
    --backbone mamba_vision_T \
    --pretrained \
    --freeze-backbone-epochs 5 \
    --backbone-lr-mult 0.1 \
    --epochs 50 \
    --batch-size 8 \
    --lr 1e-4 \
    --tag colab_run

In [ ]:
# 7. Đánh giá 3D cấp độ bệnh nhân trên tập Validation
!python -m mamba_unet.evaluate eval \
    --cache-dir /content/data/cache_1mm \
    --checkpoint checkpoints/best_model.pth \
    --split val

In [ ]:
# 8. Sao lưu kết quả và checkpoint về Google Drive an toàn
BACKUP_DIR = '/content/drive/MyDrive/DuneAI_Backups'
!mkdir -p {BACKUP_DIR}
!cp -ru mamba_unet/runs {BACKUP_DIR}/ 2>/dev/null || true
!cp -ru checkpoints {BACKUP_DIR}/ 2>/dev/null || true
print('✅ Đã sao lưu checkpoints và logs vào Google Drive!')